<a href="https://colab.research.google.com/github/romy0806/review-sentiment-agent-team/blob/main/review_sentiment_agent_team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai openai-agents datasets pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 10.5 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Key loaded:", bool(os.environ["OPENAI_API_KEY"]))

Key loaded: True


In [3]:
from openai import OpenAI
client = OpenAI()
r = client.chat.completions.create(
model = "gpt-4o-mini",
                                    messages =[{"role": "user", "content": "Say hello in five words",}])
print (r.choices[0].message.content)



Hello! How are you today?


In [4]:
from datasets import load_dataset
ds = load_dataset ("cornell-movie-review-data/rotten_tomatoes")
print (ds)
print ()
print (ds["train"] [0])
print (ds["train"] [1])

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
{'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}


In [5]:
LABELS = {0: "negative", 1: "positive"}
SYSTEM_PROMPT = "Classify the sentiment of the movie review. Answer with exactly one word: positive or negative."

def to_example(text, label_id):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
        {"role": "assistant", "content": LABELS[label_id]},
    ]}

row = ds["train"][0]
print(to_example(row["text"], row["label"]))

{'messages': [{'role': 'system', 'content': 'Classify the sentiment of the movie review. Answer with exactly one word: positive or negative.'}, {'role': 'user', 'content': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .'}, {'role': 'assistant', 'content': 'positive'}]}


In [6]:
import json
import pandas as pd

train = ds["train"].shuffle(seed=42).select(range(200))
val   = ds["validation"].shuffle(seed=42).select(range(50))
test  = ds["test"].shuffle(seed=42).select(range(100))

def write_jsonl(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(to_example(r["text"], r["label"])) + "\n")

write_jsonl("train.jsonl", train)
write_jsonl("val.jsonl", val)

pd.DataFrame({
    "text": test["text"],
    "label": [LABELS[x] for x in test["label"]],
}).to_csv("test.csv", index=False)

In [7]:
!wc -l train.jsonl val.jsonl test.csv
!head -n 2 train.jsonl
print(pd.Series(train["label"]).value_counts())
print("Overlap between train and test:", len(set(train["text"]) & set(test["text"])))

  200 train.jsonl
   50 val.jsonl
  101 test.csv
  351 total
{"messages": [{"role": "system", "content": "Classify the sentiment of the movie review. Answer with exactly one word: positive or negative."}, {"role": "user", "content": ". . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller ."}, {"role": "assistant", "content": "negative"}]}
{"messages": [{"role": "system", "content": "Classify the sentiment of the movie review. Answer with exactly one word: positive or negative."}, {"role": "user", "content": "michael moore has perfected the art of highly entertaining , self-aggrandizing , politically motivated documentary-making , and he's got as potent a topic as ever here ."}, {"role": "assistant", "content": "positive"}]}
0    105
1     95
Name: count, dtype: int64
Overlap between train and test: 0


In [8]:
!pip install -q -U transformers accelerate

from transformers import AutoTokenizer
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

train_small = ds["train"].shuffle(seed=42).select(range(1000))
val_small   = ds["validation"].shuffle(seed=42).select(range(200))

def tokenize(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=64)

train_tok = train_small.map(tokenize, batched=True)
val_tok   = val_small.map(tokenize, batched=True)

print(train_tok[0].keys())
print(tok.convert_ids_to_tokens(train_tok[0]["input_ids"])[:15])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])
['[CLS]', '.', '.', '.', 'plays', 'like', 'somebody', 'sp', '##lice', '##d', 'random', 'moments', 'of', 'a', 'chris']


In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="no", logging_steps=20, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                  eval_dataset=val_tok, compute_metrics=compute_metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.486854,0.672282,0.675000,0.659249
2,0.240534,0.436101,0.815000,0.813654
3,0.097562,0.563935,0.835000,0.834300


TrainOutput(global_step=189, training_loss=0.32094944343365056, metrics={'train_runtime': 20.4366, 'train_samples_per_second': 146.795, 'train_steps_per_second': 9.248, 'total_flos': 49675274496000.0, 'train_loss': 0.32094944343365056, 'epoch': 3.0})

In [10]:
print(pd.Series(train_small["label"]).value_counts())

0    512
1    488
Name: count, dtype: int64


In [11]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

test = ds["test"].shuffle(seed=42).select(range(100))
test_tok = test.map(tokenize, batched=True)

out = trainer.predict(test_tok)
preds = np.argmax(out.predictions, axis=-1)
labels = out.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("F1 macro:", f1_score(labels, preds, average="macro"))
print(confusion_matrix(labels, preds))
print(classification_report(labels, preds, target_names=["negative", "positive"]))

pd.set_option("display.max_colwidth", 120)
errs = pd.DataFrame({"text": test["text"], "true": labels, "pred": preds})
errs = errs[errs["true"] != errs["pred"]]
print(len(errs), "errors")
print(errs.head(10).to_string())

trainer.save_model("sentiment_model")
tok.save_pretrained("sentiment_model")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Accuracy: 0.75
F1 macro: 0.7497747973175859
[[39 12]
 [13 36]]
              precision    recall  f1-score   support

    negative       0.75      0.76      0.76        51
    positive       0.75      0.73      0.74        49

    accuracy                           0.75       100
   macro avg       0.75      0.75      0.75       100
weighted avg       0.75      0.75      0.75       100

25 errors
                                                                                                                                                                              text  true  pred
2             working from a surprisingly sensitive script co-written by gianni romoli . . . ozpetek avoids most of the pitfalls you'd expect in such a potentially sudsy set-up .     1     0
8                                                                                                                                      marinated in clichés and mawkish dialogue .     0     1
13                         

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('sentiment_model/tokenizer_config.json', 'sentiment_model/tokenizer.json')

In [12]:
from openai import OpenAI
client = OpenAI()   # needs your OPENAI_API_KEY set again after the restart

def gpt_predict(text):
    r = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0, max_tokens=3,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": text}])
    word = r.choices[0].message.content.strip().lower().strip(".!\"' ")
    return 1 if word == "positive" else 0 if word == "negative" else -1

gpt_preds = [gpt_predict(t) for t in test["text"]]
print("Invalid outputs:", sum(p == -1 for p in gpt_preds))
print("GPT accuracy:", accuracy_score(labels, gpt_preds))
print("GPT F1 macro:", f1_score(labels, gpt_preds, average="macro"))
print("DistilBERT accuracy:", accuracy_score(labels, preds))


Invalid outputs: 0
GPT accuracy: 0.9
GPT F1 macro: 0.8993558776167472
DistilBERT accuracy: 0.75


In [13]:
from transformers import pipeline

clf = pipeline("text-classification", model="sentiment_model",
               tokenizer="sentiment_model", device=0)  # use device=-1 if you're on CPU

def classify_review(text: str) -> str:
    out = clf(text, truncation=True, max_length=64)[0]
    return out["label"]

tests = [
    "A moving, beautifully acted film that stays with you.",
    "Dull, predictable, and far too long.",
    "The script is clumsy but the lead performance is wonderful.",
]
for t in tests:
    print(clf(t, truncation=True, max_length=64)[0], "|", t)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'positive', 'score': 0.9952933192253113} | A moving, beautifully acted film that stays with you.
{'label': 'negative', 'score': 0.9885808825492859} | Dull, predictable, and far too long.
{'label': 'negative', 'score': 0.9588178396224976} | The script is clumsy but the lead performance is wonderful.


In [14]:
from agents import Agent, Runner, function_tool

@function_tool
def classify_review(review: str) -> str:
    """Classify one review as positive or negative using the fine-tuned DistilBERT model."""
    out = clf(review, truncation=True, max_length=64)[0]
    return f"{out['label']} (confidence {out['score']:.2f})"

agent = Agent(
    name="Review classifier",
    model="gpt-4o-mini",
    instructions=("You classify customer reviews. Call the classify_review tool once for each "
                  "review the user gives you. Report each label and confidence. "
                  "Never decide a label yourself."),
    tools=[classify_review],
)

reviews = """Classify these reviews:
1. A moving, beautifully acted film that stays with you.
2. Dull, predictable, and far too long.
3. The script is clumsy but the lead performance is wonderful."""

result = await Runner.run(agent, reviews)
print(result.final_output)
print([type(i).__name__ for i in result.new_items])

Here are the classifications for the reviews:

1. **"A moving, beautifully acted film that stays with you."** - **Positive** (Confidence: 1.00)
2. **"Dull, predictable, and far too long."** - **Negative** (Confidence: 0.99)
3. **"The script is clumsy but the lead performance is wonderful."** - **Negative** (Confidence: 0.96)
['ToolCallItem', 'ToolCallItem', 'ToolCallItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'MessageOutputItem']


In [15]:
summary_agent = Agent(
    name="Summary agent",
    model="gpt-4o-mini",
    instructions=("The reviews in this conversation have already been classified. Write a 3-sentence "
                  "insight note for a product manager: the counts of positive and negative, one theme "
                  "you notice, and a caution that the classifier is a small model and the sample is tiny. "
                  "Do not reclassify anything."),
)

classifier_agent = Agent(
    name="Classifier agent",
    model="gpt-4o-mini",
    instructions=("Call classify_review once for each review. When every review has been classified, "
                  "hand off to the Summary agent."),
    tools=[classify_review],
    handoffs=[summary_agent],
)

triage_agent = Agent(
    name="Triage agent",
    model="gpt-4o-mini",
    instructions="You receive a batch of customer reviews. Hand off to the Classifier agent.",
    handoffs=[classifier_agent],
)

reviews = """Here are some reviews:
1. A moving, beautifully acted film that stays with you.
2. Dull, predictable, and far too long.
3. The script is clumsy but the lead performance is wonderful.
4. Marinated in clichés and mawkish dialogue.
5. A surprisingly sensitive script that avoids most of the pitfalls of its genre."""

result = await Runner.run(triage_agent, reviews)
print(result.final_output)
print("Last agent:", result.last_agent.name)
print([type(i).__name__ for i in result.new_items])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The review analysis reveals a count of three positive and two negative reviews. A notable theme is the contrasting views on the script's quality, with some praising the lead performance while others criticize the writing as clichéd. However, it's important to note that the classifier used is a small model and the review sample size is limited, which may affect the reliability of these insights.
Last agent: Summary agent
['HandoffCallItem', 'HandoffOutputItem', 'ToolCallItem', 'ToolCallItem', 'ToolCallItem', 'ToolCallItem', 'ToolCallItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'ToolCallOutputItem', 'HandoffCallItem', 'HandoffOutputItem', 'MessageOutputItem']
